In [7]:
import os
os.chdir(r'D:\8th Semester\Machine Learning Lab\lab no 12')

In [44]:
pip install --upgrade tensorflow keras

   ---------------------------------------- 0.0/350.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/350.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/350.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/350.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/350.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/350.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/350.8 MB ? eta -:--:--
   ---------------------------------------- 0.5/350.8 MB 262.1 kB/s eta 0:22:17
   ---------------------------------------- 0.5/350.8 MB 262.1 kB/s eta 0:22:17
   ---------------------------------------- 0.5/350.8 MB 262.1 kB/s eta 0:22:17
   ---------------------------------------- 0.5/350.8 MB 262.1 kB/s eta 0:22:17
   ---------------------------------------- 0.5/350.8 MB 262.1 kB/s eta 0:22:17
   ---------------------------------------- 0.8/350.8 MB 260.1 kB/s eta 0:22:27
   --------------------

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\Users\\Shafiq\\.conda\\envs\\DSP\\Lib\\site-packages\\tensorflow\\compiler\\mlir\\quantization\\tensorflow\\python\\pywrap_quantize_model.pyd'
Consider using the `--user` option or check the permissions.



In [8]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [9]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [10]:
def create_lstm():
    input_data = Input(shape=(time_steps, num_features))
    lstm_layer1 = LSTM(8, return_sequences=True)(input_data)
    lstm_layer2 = LSTM(20)(lstm_layer1)
    x = Flatten()(lstm_layer2)
    output_data = Dense(1)(x)
    model = Model(input_data, output_data)
    return model

In [11]:
model1 = create_lstm()
model1.summary()


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 24, 21)]          0         
                                                                 
 lstm (LSTM)                 (None, 24, 8)             960       
                                                                 
 lstm_1 (LSTM)               (None, 20)                2320      
                                                                 
 flatten (Flatten)           (None, 20)                0         
                                                                 
 dense (Dense)               (None, 1)                 21        
                                                                 
Total params: 3301 (12.89 KB)
Trainable params: 3301 (12.89 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [12]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


In [13]:
checkpoints = r'D:\8th Semester\Machine Learning Lab\lab no 12-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'D:\8th Semester\Machine Learning Lab\lab no 12'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [14]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

In [15]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =create_lstm()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


In [23]:
import os
path_dataset =r'D:\8th Semester\Machine Learning Lab\data parisr'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\Shafiq\.conda\envs\DSP\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

In [24]:
time_steps=24
num_features=21

In [25]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.005985260009765625 sec


In [26]:
epochs = 60
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,verbose = verbose)

Epoch 1/60


26/27 [===========================>..] - ETA: 0s - loss: 0.2553 - mae: 0.2553 - mape: 101.3264
Epoch 1: val_loss improved from inf to 0.11589, saving model to D:\8th Semester\Machine Learning Lab\lab no 12-cp-0001-loss0.12.h5
27/27 [==============================] - 8s 87ms/step - loss: 0.2551 - mae: 0.2551 - mape: 101.6861 - val_loss: 0.1159 - val_mae: 0.1159 - val_mape: 52.6595
Epoch 2/60
 3/27 [==>...........................] - ETA: 0s - loss: 0.1563 - mae: 0.1563 - mape: 183.6992

c:\Users\Shafiq\.conda\envs\DSP\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


26/27 [===========================>..] - ETA: 0s - loss: 0.1231 - mae: 0.1231 - mape: 87.5775
Epoch 2: val_loss improved from 0.11589 to 0.07939, saving model to D:\8th Semester\Machine Learning Lab\lab no 12-cp-0002-loss0.08.h5
27/27 [==============================] - 1s 45ms/step - loss: 0.1231 - mae: 0.1231 - mape: 87.4286 - val_loss: 0.0794 - val_mae: 0.0794 - val_mape: 28.7532
Epoch 3/60
26/27 [===========================>..] - ETA: 0s - loss: 0.0818 - mae: 0.0818 - mape: 49.1274
Epoch 3: val_loss improved from 0.07939 to 0.04171, saving model to D:\8th Semester\Machine Learning Lab\lab no 12-cp-0003-loss0.04.h5


c:\Users\Shafiq\.conda\envs\DSP\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 34ms/step - loss: 0.0816 - mae: 0.0816 - mape: 49.0218 - val_loss: 0.0417 - val_mae: 0.0417 - val_mape: 15.8602
Epoch 4/60
27/27 [==============================] - ETA: 0s - loss: 0.0580 - mae: 0.0580 - mape: 25.9355
Epoch 4: val_loss improved from 0.04171 to 0.03983, saving model to D:\8th Semester\Machine Learning Lab\lab no 12-cp-0004-loss0.04.h5
27/27 [==============================] - 1s 29ms/step - loss: 0.0580 - mae: 0.0580 - mape: 25.9355 - val_loss: 0.0398 - val_mae: 0.0398 - val_mape: 13.3458


c:\Users\Shafiq\.conda\envs\DSP\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Epoch 5/60
27/27 [==============================] - ETA: 0s - loss: 0.0527 - mae: 0.0527 - mape: 23.1663
Epoch 5: val_loss did not improve from 0.03983
27/27 [==============================] - 1s 29ms/step - loss: 0.0527 - mae: 0.0527 - mape: 23.1663 - val_loss: 0.0413 - val_mae: 0.0413 - val_mape: 13.6721
Epoch 6/60
27/27 [==============================] - ETA: 0s - loss: 0.0450 - mae: 0.0450 - mape: 20.8328
Epoch 6: val_loss did not improve from 0.03983
27/27 [==============================] - 1s 27ms/step - loss: 0.0450 - mae: 0.0450 - mape: 20.8328 - val_loss: 0.0406 - val_mae: 0.0406 - val_mape: 13.3676
Epoch 7/60
25/27 [==========================>...] - ETA: 0s - loss: 0.0420 - mae: 0.0420 - mape: 19.8421
Epoch 7: val_loss did not improve from 0.03983
27/27 [==============================] - 1s 39ms/step - loss: 0.0422 - mae: 0.0422 - mape: 20.1070 - val_loss: 0.0451 - val_mae: 0.0451 - val_mape: 15.5389
Epoch 8/60
26/27 [===========================>..] - ETA: 0s - loss: 0.0394 -

c:\Users\Shafiq\.conda\envs\DSP\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 44ms/step - loss: 0.0207 - mae: 0.0207 - mape: 8.6238 - val_loss: 0.0295 - val_mae: 0.0295 - val_mape: 10.2906
Epoch 45/60
26/27 [===========================>..] - ETA: 0s - loss: 0.0207 - mae: 0.0207 - mape: 8.9117
Epoch 45: val_loss did not improve from 0.02948
27/27 [==============================] - 1s 36ms/step - loss: 0.0207 - mae: 0.0207 - mape: 9.0699 - val_loss: 0.0508 - val_mae: 0.0508 - val_mape: 17.3385
Epoch 46/60
27/27 [==============================] - ETA: 0s - loss: 0.0212 - mae: 0.0212 - mape: 9.2127
Epoch 46: val_loss did not improve from 0.02948
27/27 [==============================] - 1s 34ms/step - loss: 0.0212 - mae: 0.0212 - mape: 9.2127 - val_loss: 0.0405 - val_mae: 0.0405 - val_mape: 14.2342
Epoch 47/60
26/27 [===========================>..] - ETA: 0s - loss: 0.0187 - mae: 0.0187 - mape: 7.6222
Epoch 47: val_loss did not improve from 0.02948
27/27 [==============================] - 1s 36ms/step - loss: 0.0187 - mae: 

In [36]:

model = load_model(r'D:\8th Semester\Machine Learning Lab\lab no 12-cp-0001-loss0.12.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 3s 3s/step
Mean Absolute Error (MAE): 1521.72
Median Absolute Error (MedAE): 1456.17
Mean Squared Error (MSE): 2362267.36
Root Mean Squared Error (RMSE): 1536.97
Mean Absolute Percentage Error (MAPE): 9.7 %
Median Absolute Percentage Error (MDAPE): 9.26 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


In [40]:
checkpoints = r'D:\8th Semester\Machine Learning Lab\lab no 12-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
model=r'D:\8th Semester\Machine Learning Lab\lab no 12-cp-0004-loss0.04.h5'
start_epoch= 34

In [41]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] loading D:\8th Semester\Machine Learning Lab\lab no 12-cp-0004-loss0.04.h5...
[INFO] old learning rate: 0.0010000000474974513
[INFO] new learning rate: 9.999999747378752e-05


In [42]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
25/27 [==========================>...] - ETA: 0s - loss: 0.0486 - mae: 0.0486 - mape: 23.3713
Epoch 1: val_loss improved from inf to 0.03791, saving model to D:\8th Semester\Machine Learning Lab\lab no 12-cp-0001-loss0.04.h5


c:\Users\Shafiq\.conda\envs\DSP\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 6s 86ms/step - loss: 0.0491 - mae: 0.0491 - mape: 23.2852 - val_loss: 0.0379 - val_mae: 0.0379 - val_mape: 12.8431
Epoch 2/10
24/27 [=========================>....] - ETA: 0s - loss: 0.0487 - mae: 0.0487 - mape: 23.1265
Epoch 2: val_loss did not improve from 0.03791
27/27 [==============================] - 1s 28ms/step - loss: 0.0486 - mae: 0.0486 - mape: 22.8539 - val_loss: 0.0392 - val_mae: 0.0392 - val_mape: 13.1536
Epoch 3/10
27/27 [==============================] - ETA: 0s - loss: 0.0480 - mae: 0.0480 - mape: 22.3581
Epoch 3: val_loss did not improve from 0.03791
27/27 [==============================] - 1s 30ms/step - loss: 0.0480 - mae: 0.0480 - mape: 22.3581 - val_loss: 0.0400 - val_mae: 0.0400 - val_mape: 13.3597
Epoch 4/10
27/27 [==============================] - ETA: 0s - loss: 0.0475 - mae: 0.0475 - mape: 22.4914
Epoch 4: val_loss did not improve from 0.03791
27/27 [==============================] - 1s 33ms/step - loss: 0.0475 - mae: 

In [43]:

model = load_model(r'D:\8th Semester\Machine Learning Lab\lab no 12-cp-0001-loss0.12.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 1s 1s/step
Mean Absolute Error (MAE): 1521.72
Median Absolute Error (MedAE): 1456.17
Mean Squared Error (MSE): 2362267.36
Root Mean Squared Error (RMSE): 1536.97
Mean Absolute Percentage Error (MAPE): 9.7 %
Median Absolute Percentage Error (MDAPE): 9.26 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)
